# 03 · RAG sobre papers academicos con NVIDIA NIM

Workshop RAG · SI3003 · **Daniel Muñeton**

RAG sobre papers fundamentales. Base: *Attention Is All You Need* (Vaswani et
al., 2017) y *Retrieval-Augmented Generation...* (Lewis et al., 2020). Reto C2
agrega un tercer paper sobre **HNSW** (Malkov & Yashunin, 2016).

Stack: MarkItDown + LangChain + NVIDIA NIM + Qdrant.

Tareas implementadas:
- **B1** · Ejecutar el RAG y evaluar con ground truth.
- **B2** · Experimentos con `chunk_size`, `k` y `score_threshold`
  (funciones `build_store` y `build_chain`).
- **B3** · Prompt estricto + preguntas trampa.
- **B4** · Preguntas en espanol sobre papers en ingles.
- **B5** · Quiz propio con ground truth.
- **C2** · Tercer paper (HNSW) en el diccionario `PAPERS`.
- **C3** · App Gradio con filtro por paper (metadata).
- **C5** · `search_type="mmr"` vs similitud pura.
- **C6** · Persistencia del indice en disco (`QdrantClient(path=...)`).

## Parte 0 · Instalar y configurar

In [ ]:
%pip install -qU "markitdown[pdf]" langchain langchain-core langchain-text-splitters \
    langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client requests gradio

In [ ]:
import os
from getpass import getpass
try:
    from google.colab import userdata  # type: ignore
    key = userdata.get("NVIDIA_API_KEY")
    if key:
        os.environ["NVIDIA_API_KEY"] = key
except Exception:
    pass
if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass("NVIDIA_API_KEY: ")

from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

GENERATION_MODEL = "nvidia/nemotron-3-super-120b-a12b"
EMBEDDING_MODEL  = "nvidia/nemotron-3-embed-1b"
COLLECTION       = "academic_papers_nvidia"
print("\u2705 Configuracion NVIDIA NIM lista")

## Parte 1 · Descargar y convertir los PDFs  _(incluye C2)_

`PAPERS` es un diccionario `nombre_archivo -> url`. **C2:** agrego un tercer
paper (HNSW) simplemente anadiendo una entrada al diccionario.

In [ ]:
import requests
from pathlib import Path
from markitdown import MarkItDown

PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf":       "https://arxiv.org/pdf/2005.11401",
    # C2 · tercer paper: HNSW (Malkov & Yashunin, 2016)
    "hnsw_paper.pdf":      "https://arxiv.org/pdf/1603.09320",
}
TITULOS = {
    "attention_paper.pdf": "Attention Is All You Need",
    "rag_paper.pdf":       "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
    "hnsw_paper.pdf":      "Efficient and robust approximate nearest neighbor search using HNSW",
}

PAPERS_DIR = Path("papers")
PAPERS_DIR.mkdir(exist_ok=True)

def descargar(nombre, url):
    destino = PAPERS_DIR / nombre
    if destino.exists() and destino.stat().st_size > 10_000:
        print(f"\u21aa Ya existe: {nombre}")
        return destino
    r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    if not r.content.startswith(b"%PDF") or len(r.content) < 10_000:
        raise RuntimeError(f"La descarga de {nombre} no parece un PDF valido.")
    destino.write_bytes(r.content)
    print(f"\u2b07 Descargado: {nombre} ({len(r.content)/1e6:.2f} MB)")
    return destino

md = MarkItDown()
textos = {}
for nombre, url in PAPERS.items():
    ruta = descargar(nombre, url)
    textos[nombre] = md.convert(str(ruta)).text_content
    print(f"\u2705 {nombre:>20} -> {len(textos[nombre])} caracteres")

## Parte 2 · Document y chunking  _(Tarea B2)_

`pdf_to_documents` parte el texto en fragmentos, descarta los de menos de 30
caracteres y guarda `title`, `source` y `chunk_index`. Los valores 900/120 son
"un punto de partida didactico" que cuestionare en B2.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def pdf_to_documents(nombre, texto, chunk_size=900, chunk_overlap=120):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "],
    )
    docs = []
    for i, frag in enumerate(splitter.split_text(texto)):
        if len(frag.strip()) < 30:
            continue
        docs.append(Document(page_content=frag, metadata={
            "title": TITULOS[nombre], "source": nombre, "chunk_index": i,
        }))
    return docs

def construir_documentos(chunk_size=900, chunk_overlap=120):
    todos = []
    for nombre, texto in textos.items():
        d = pdf_to_documents(nombre, texto, chunk_size, chunk_overlap)
        print(f"{TITULOS[nombre][:30]:<30} -> {len(d)} chunks")
        todos.extend(d)
    print(f"Total -> {len(todos)} chunks")
    return todos

all_docs = construir_documentos()
print("Ejemplo de metadatos:", all_docs[0].metadata)

## Parte 3 · Embeddings e indice Qdrant

In [ ]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")

def build_store(docs, collection=COLLECTION, client=None):
    """B2: crea un QdrantVectorStore en memoria (o con el cliente dado) y lo
    puebla con los docs. Devuelve (vectorstore, dim)."""
    dim = len(embeddings.embed_query("prueba de dimension"))
    client = client or QdrantClient(":memory:")
    if not client.collection_exists(collection):
        client.create_collection(
            collection_name=collection,
            vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
        )
    vs = QdrantVectorStore(client=client, collection_name=collection, embedding=embeddings)
    vs.add_documents(docs)
    return vs, dim

vectorstore, dim = build_store(all_docs)
print(f"\u2705 {len(all_docs)} chunks indexados en Qdrant")
print(f"\u2705 Dimension de embeddings: {dim}")

## Parte 4 · Pipeline con LCEL  _(Tareas B2, B3 y reto C5)_

**B3 (prompt):** el system prompt exige usar solo los extractos, reconocer
cuando no hay evidencia, responder en el idioma de la pregunta y nombrar el
paper que respalda la respuesta.

**B2:** `build_chain` permite variar `k` y `score_threshold`.

**C5:** parametro `search_type` para alternar entre `similarity_score_threshold`
y `mmr`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

SYSTEM_PROMPT_PAPERS = """You are a precise research assistant. Answer the
question using ONLY the excerpts provided below. Rules:
- If the excerpts do not contain the answer, say clearly that the papers do not
  provide that information. Do not use outside knowledge and do not invent.
- Answer in the SAME language as the question.
- Mention which paper (by title) supports your answer.

Excerpts:
{context}

Question: {question}
Answer:"""

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['title']} | {d.metadata['source']}] {d.page_content}"
                        for d in docs)

def build_chain(vs, k=5, score_threshold=0.25, search_type="similarity_score_threshold",
                return_sources=False):
    """B2/C5: arma retriever + cadena LCEL con parametros configurables."""
    if search_type == "mmr":
        retriever = vs.as_retriever(search_type="mmr",
                                    search_kwargs={"k": k, "fetch_k": max(20, k * 4)})
    else:
        retriever = vs.as_retriever(search_type="similarity_score_threshold",
                                    search_kwargs={"k": k, "score_threshold": score_threshold})
    llm = ChatNVIDIA(model=GENERATION_MODEL, temperature=0.2, max_completion_tokens=700)
    prompt = ChatPromptTemplate.from_template(SYSTEM_PROMPT_PAPERS)
    core = {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
    if return_sources:
        return core, retriever
    return core

chain = build_chain(vectorstore)
print(chain.invoke("What is the Transformer architecture?"))

## Parte 5 · Quiz con ground truth  _(Tareas B1, B4, B5 y Bonus)_

### B1 · Preguntas de referencia (ingles) con su ground truth
Para cada pregunta evaluo tres cosas: correcta (si/parcial/no), paper correcto
y completitud.

In [ ]:
QUIZ_B1 = [
    ("How many attention heads does the base Transformer use?",
     "8 heads, con d_k = d_v = 64 (attention_paper)."),
    ("What are the two RAG model variants?",
     "RAG-Sequence y RAG-Token (rag_paper)."),
    ("What is the dimension d_model of the base Transformer?",
     "d_model = 512 (attention_paper)."),
    ("What retriever does RAG use?",
     "Un recuperador denso de pasajes DPR sobre Wikipedia (rag_paper)."),
    ("How many layers (N) does the encoder of the base Transformer have?",
     "N = 6 capas identicas (attention_paper)."),
]

for pregunta, ground_truth in QUIZ_B1:
    print("P:", pregunta)
    print("Modelo:", chain.invoke(pregunta).strip())
    print("Ground truth:", ground_truth)
    print("-" * 70)

### B3 · Preguntas trampa: ¿el RAG admite que no sabe?
Preguntas cuya respuesta NO esta en los papers. El prompt debe hacer que el
modelo reconozca la ausencia de evidencia.

In [ ]:
PREGUNTAS_TRAMPA = [
    "What is the training cost in US dollars of GPT-4?",
    "Who won the 2022 FIFA World Cup according to these papers?",
    "What learning rate did BERT use for fine-tuning on SQuAD?",
]
for p in PREGUNTAS_TRAMPA:
    print("P:", p)
    print("R:", chain.invoke(p).strip())
    print("-" * 70)

### B4 · Preguntas en espanol sobre papers en ingles
El prompt pide responder en el idioma de la pregunta. Verifico que responda en
espanol aunque los papers esten en ingles.

In [ ]:
PREGUNTAS_ES = [
    "¿Cuantas cabezas de atencion usa el Transformer base?",
    "¿Que dos variantes del modelo RAG existen y en que se diferencian?",
    "¿Que mecanismo usa el Transformer para incorporar el orden de las palabras?",
]
for p in PREGUNTAS_ES:
    print("P:", p)
    print("R:", chain.invoke(p).strip())
    print("-" * 70)

### B5 · Mi propio quiz con ground truth
Cinco preguntas nuevas escritas por mi + una **pregunta bonus que cruza ambos
papers**.

In [ ]:
MI_QUIZ = [
    ("What normalization and residual structure does each Transformer sub-layer use?",
     "Cada sub-capa usa conexion residual seguida de layer normalization: LayerNorm(x + Sublayer(x)) (attention_paper)."),
    ("Why does the Transformer scale the dot products by 1/sqrt(d_k)?",
     "Para evitar que los productos punto crezcan en magnitud y empujen al softmax a regiones de gradiente muy pequeno (attention_paper)."),
    ("What are the parametric and non-parametric components of RAG?",
     "El componente parametrico es un seq2seq (BART) y el no parametrico es un indice denso de Wikipedia accedido por DPR (rag_paper)."),
    ("How does RAG-Token differ from RAG-Sequence at generation time?",
     "RAG-Token puede usar un documento distinto por token generado, mientras RAG-Sequence usa el mismo documento para toda la secuencia (rag_paper)."),
    ("What problem of large parametric LMs does RAG aim to mitigate?",
     "La dificultad para acceder y actualizar conocimiento factual y la tendencia a alucinar; RAG agrega memoria no parametrica actualizable (rag_paper)."),
]

# Bonus · pregunta que cruza ambos papers
BONUS = ("Both papers build on neural architectures: which specific architecture "
         "does RAG use as its generator, and is it based on the Transformer?")

print("### Mi quiz ###")
for pregunta, gt in MI_QUIZ:
    print("P:", pregunta)
    print("Modelo:", chain.invoke(pregunta).strip())
    print("Ground truth:", gt)
    print("-" * 70)

print("### Bonus (cruza papers, muestra fuentes) ###")
core, retriever = build_chain(vectorstore, return_sources=True)
print("R:", core.invoke(BONUS).strip())
for d in retriever.invoke(BONUS):
    print("  fuente:", d.metadata["title"], "|", d.metadata["source"])

## Reto C3 · App Gradio para los papers con filtro por paper

Interfaz Gradio con un desplegable que filtra la recuperacion por `source`
(metadata), ademas de la opcion "Todos".

In [ ]:
import gradio as gr
from qdrant_client import models as qmodels

OPCIONES = {"Todos los papers": None, **{TITULOS[n]: n for n in PAPERS}}

def responder_con_filtro(pregunta, paper_elegido):
    source = OPCIONES.get(paper_elegido)
    filtro = None
    if source is not None:
        filtro = qmodels.Filter(must=[qmodels.FieldCondition(
            key="metadata.source", match=qmodels.MatchValue(value=source))])
    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 5, "score_threshold": 0.25, "filter": filtro},
    )
    docs = retriever.invoke(pregunta)
    if not docs:
        return "Sin evidencia en el (los) paper(s) seleccionado(s).", ""
    llm = ChatNVIDIA(model=GENERATION_MODEL, temperature=0.2, max_completion_tokens=700)
    prompt = ChatPromptTemplate.from_template(SYSTEM_PROMPT_PAPERS)
    cadena = (lambda _: {"context": format_docs(docs), "question": pregunta}) | prompt | llm | StrOutputParser()
    respuesta = cadena.invoke(None)
    fuentes = "\n".join(f"- {d.metadata['title']} ({d.metadata['source']})" for d in docs)
    return respuesta, fuentes

with gr.Blocks(title="RAG de papers con filtro") as demo_papers:
    gr.Markdown("# RAG sobre papers · filtro por paper (C3)")
    q = gr.Textbox(label="Pregunta")
    sel = gr.Dropdown(choices=list(OPCIONES.keys()), value="Todos los papers", label="Filtrar por paper")
    b = gr.Button("Preguntar", variant="primary")
    out = gr.Markdown()
    src = gr.Markdown()
    b.click(responder_con_filtro, inputs=[q, sel], outputs=[out, src])

demo_papers.queue().launch(share=True, debug=True, show_error=True)

## Reto C5 · MMR frente a similitud pura

Comparo las fuentes recuperadas por similitud pura vs MMR (que penaliza la
redundancia). Observa si MMR trae chunks mas diversos (de distintos papers o
secciones) para la misma pregunta.

In [ ]:
pregunta_c5 = "How do these papers combine neural networks with attention or retrieval?"

print("=== Similitud pura (score_threshold) ===")
_, r_sim = build_chain(vectorstore, k=5, return_sources=True)
for d in r_sim.invoke(pregunta_c5):
    print("  ", d.metadata["source"], "| chunk", d.metadata["chunk_index"], "|", d.page_content[:70].replace("\n", " "))

print("\n=== MMR ===")
_, r_mmr = build_chain(vectorstore, k=5, search_type="mmr", return_sources=True)
for d in r_mmr.invoke(pregunta_c5):
    print("  ", d.metadata["source"], "| chunk", d.metadata["chunk_index"], "|", d.page_content[:70].replace("\n", " "))

## Reto C6 · Persistencia del indice en disco

En vez de `QdrantClient(":memory:")`, uso `QdrantClient(path="./qdrant_data")`.
Si la coleccion ya existe, la reutilizo sin reindexar; si no, la creo y mido el
tiempo. Tras reiniciar el runtime, esta celda debe responder sin reindexar.

In [ ]:
import time
from qdrant_client import QdrantClient

PERSIST_PATH = "./qdrant_papers"
COLLECTION_DISK = "academic_papers_disk"

client_disk = QdrantClient(path=PERSIST_PATH)
existe = client_disk.collection_exists(COLLECTION_DISK)
print("¿Ya existe la coleccion en disco?", existe)

t0 = time.time()
if not existe:
    vs_disk, _ = build_store(all_docs, collection=COLLECTION_DISK, client=client_disk)
    print(f"Indexado desde cero en {time.time()-t0:.2f} s")
else:
    from langchain_qdrant import QdrantVectorStore
    vs_disk = QdrantVectorStore(client=client_disk, collection_name=COLLECTION_DISK, embedding=embeddings)
    print(f"Reutilizada sin reindexar en {time.time()-t0:.2f} s")

chain_disk = build_chain(vs_disk)
print(chain_disk.invoke("What is multi-head attention?").strip())

## Reflexion final  _(Tarea R1)_

Las cinco preguntas de reflexion estan respondidas con mis datos en
`RESPUESTAS.md` (seccion R1). Resumen de lo observado:

1. El `score_threshold` controla el equilibrio entre cobertura y precision:
   umbrales altos hacen que el RAG diga "no se" mas seguido.
2. `chunk_size` afecta si un dato cabe entero en un chunk; trozos muy pequenos
   fragmentan ideas, trozos muy grandes diluyen la relevancia.
3. El prompt es decisivo para que el modelo admita falta de evidencia (B3).
4. MMR aporta diversidad cuando varios chunks dicen casi lo mismo (C5).
5. Persistir en disco (C6) evita recomputar embeddings en cada reinicio.